In [ ]:
# INIT SPLASH FIDÉLITÉ : scraping complet (à lancer une seule fois)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import os

# Configuration
WEBDRIVER_PATH = os.getenv("SPLASH_FIDELITE_DRIVER_PATH", r"C:\path\to\geckodriver.exe")
EMAIL = os.getenv("SPLASH_FIDELITE_EMAIL", "your_email_here")
PASSWORD = os.getenv("SPLASH_FIDELITE_PASSWORD", "your_password_here")
URL = "https://manager-loyalty.splash360.fr/#/welcome"
FICHIER_CSV_INIT = "BDD_Client_SplashFidelite.csv"

service = Service(WEBDRIVER_PATH)
driver = webdriver.Firefox(service=service)
driver.get(URL)


def get_restaurant(magasin):
    magasin = magasin.strip().upper()
    if "CHICKEN" in magasin:
        return "Chicken Street Pontault Combault"
    if "BANGKOK" in magasin:
        return "Bangkok Factory Pontault Combault"
    return "Inconnu"


# Connexion
WebDriverWait(driver, 15).until(
    EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Login')]/parent::button"))
).click()
WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.ID, "username"))
).send_keys(EMAIL)
driver.find_element(By.ID, "password").send_keys(PASSWORD)
driver.find_element(By.ID, "kc-login").click()

# Accès clients
WebDriverWait(driver, 20).until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, '#/mobileusers')]"))
).click()
WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.CLASS_NAME, "datatable-body"))
)
time.sleep(2)

# Scraping
data = []

while True:
    print("Récupération page...")
    rows = driver.find_elements(By.CLASS_NAME, "datatable-row-wrapper")

    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "datatable-body-cell")
        row_data = [cell.text for cell in cells]

        if len(row_data) >= 7:
            restaurant = get_restaurant(row_data[2])

            data.append([
                row_data[0],
                row_data[1],
                restaurant,
                row_data[3],
                row_data[4],
                row_data[5],
                row_data[6],
                "splash-fidélité"
            ])

    try:
        current_page = driver.find_element(By.CSS_SELECTOR, "li.pages.active.ng-star-inserted").text.strip()
        next_button = driver.find_element(By.XPATH, "//a[@aria-label='go to next page']")

        if next_button.is_enabled():
            next_button.click()
            time.sleep(2)
            new_page = driver.find_element(By.CSS_SELECTOR, "li.pages.active.ng-star-inserted").text.strip()
            if new_page == current_page:
                break
        else:
            break

    except Exception:
        break

driver.quit()

columns = [
    "Téléphone",
    "Prénom",
    "Restaurant",
    "Création",
    "Dernière Visite",
    "Nombre de visites",
    "Nombre de points",
    "Plateforme"
]

df = pd.DataFrame(data, columns=columns)
df.to_csv(FICHIER_CSV_INIT, index=False, encoding="utf-8")
print(f"Base initiale enregistrée dans : {FICHIER_CSV_INIT} ({len(df)} lignes)")

In [ ]:
# SCRAPING RÉGULIER SPLASH FIDÉLITÉ : récupération des nouvelles données

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import os

# Configuration
WEBDRIVER_PATH = os.getenv("SPLASH_FIDELITE_DRIVER_PATH", r"C:\path\to\geckodriver.exe")
EMAIL = os.getenv("SPLASH_FIDELITE_EMAIL", "your_email_here")
PASSWORD = os.getenv("SPLASH_FIDELITE_PASSWORD", "your_password_here")
URL = "https://manager-loyalty.splash360.fr/#/welcome"
FICHIER_CSV_TEMP = "SplashFidelite_Temp.csv"

service = Service(WEBDRIVER_PATH)
driver = webdriver.Firefox(service=service)
driver.get(URL)


def get_restaurant(magasin):
    magasin = magasin.strip().upper()
    if "CHICKEN" in magasin:
        return "Chicken Street Pontault Combault"
    if "BANGKOK" in magasin:
        return "Bangkok Factory Pontault Combault"
    return "Inconnu"


# Connexion
WebDriverWait(driver, 15).until(
    EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), 'Login')]/parent::button"))
).click()
WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.ID, "username"))
).send_keys(EMAIL)
driver.find_element(By.ID, "password").send_keys(PASSWORD)
driver.find_element(By.ID, "kc-login").click()

# Accès clients
WebDriverWait(driver, 20).until(
    EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, '#/mobileusers')]"))
).click()
WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.CLASS_NAME, "datatable-body"))
)
time.sleep(2)

# Scraping
data = []

while True:
    print("Récupération page...")
    rows = driver.find_elements(By.CLASS_NAME, "datatable-row-wrapper")

    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "datatable-body-cell")
        row_data = [cell.text for cell in cells]

        if len(row_data) >= 7:
            restaurant = get_restaurant(row_data[2])

            data.append([
                row_data[0],
                row_data[1],
                restaurant,
                row_data[3],
                row_data[4],
                row_data[5],
                row_data[6],
                "splash-fidélité"
            ])

    try:
        current_page = driver.find_element(By.CSS_SELECTOR, "li.pages.active.ng-star-inserted").text.strip()
        next_button = driver.find_element(By.XPATH, "//a[@aria-label='go to next page']")

        if next_button.is_enabled():
            next_button.click()
            time.sleep(2)
            new_page = driver.find_element(By.CSS_SELECTOR, "li.pages.active.ng-star-inserted").text.strip()
            if new_page == current_page:
                print("Dernière page atteinte.")
                break
        else:
            print("Dernière page atteinte.")
            break

    except Exception as e:
        print("Fin atteinte ou erreur :", e)
        break

# Sauvegarde des données temporaires
driver.quit()

columns = [
    "Téléphone",
    "Prénom",
    "Restaurant",
    "Création",
    "Dernière Visite",
    "Nombre de visites",
    "Nombre de points",
    "Plateforme"
]

df = pd.DataFrame(data, columns=columns)
df.to_csv(FICHIER_CSV_TEMP, index=False, encoding="utf-8")
print(f"Nouvelles données sauvegardées dans : {FICHIER_CSV_TEMP} ({len(df)} lignes)")

In [ ]:
# FUSION SPLASH FIDÉLITÉ : fusionner la base existante et les nouvelles données

import pandas as pd

FICHIER_CSV_BASE = "BDD_Client_SplashFidelite.csv"
FICHIER_CSV_TEMP = "SplashFidelite_Temp.csv"

df_old = pd.read_csv(FICHIER_CSV_BASE)
df_new = pd.read_csv(FICHIER_CSV_TEMP)

# Correction automatique des valeurs manquantes
df_old["Restaurant"] = df_old["Restaurant"].replace("Inconnu", "Al Forno Pontault Combault")
df_new["Restaurant"] = df_new["Restaurant"].replace("Inconnu", "Al Forno Pontault Combault")

# Harmoniser les colonnes
for col in df_old.columns:
    if col not in df_new.columns:
        df_new[col] = None

df_new = df_new[df_old.columns]

# Fusion + dédoublonnage
df_combined = pd.concat([df_old, df_new], ignore_index=True)

colonnes_infos = [col for col in ["Prénom", "Téléphone", "Email", "Restaurant"] if col in df_combined.columns]

df_combined["infos"] = df_combined[colonnes_infos].notnull().sum(axis=1)
df_combined = df_combined.sort_values(by="infos", ascending=False)
df_combined = df_combined.drop_duplicates(subset="Téléphone", keep="first").drop(columns="infos")

df_combined.to_csv(FICHIER_CSV_BASE, index=False, encoding="utf-8")
print(f"Fusion terminée. Base mise à jour : {FICHIER_CSV_BASE}")